In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_60K_278467_unopt_pbesol_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [7]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-76.30218129 169.62690273 -78.56677332]
 [171.69409189  74.01931836  27.97617649]
 [  9.91666475 -24.32411597 -58.70339189]]

17O2 sigma:
 [[ -76.30218129 -169.62690273   78.56677332]
 [-171.69409189   74.01931836   27.97617649]
 [  -9.91666475  -24.32411597  -58.70339189]]

17O3 sigma:
 [[-76.30218129 169.62690273  78.56677332]
 [171.69409189  74.01931836 -27.97617649]
 [ -9.91666475  24.32411597 -58.70339189]]

17O4 sigma:
 [[ -76.30218129 -169.62690273  -78.56677332]
 [-171.69409189   74.01931836  -27.97617649]
 [   9.91666475   24.32411597  -58.70339189]]

17O5 sigma:
 [[ -62.84783893  211.51902361   60.92122523]
 [ 182.27829083  116.9822964   -85.07839589]
 [  20.31785642  -45.48587933 -174.98288596]]

17O6 sigma:
 [[ -62.84783893 -211.51902361  -60.92122523]
 [-182.27829083  116.9822964   -85.07839589]
 [ -20.31785642  -45.48587933 -174.98288596]]

17O7 sigma:
 [[ -62.84783893  211.51902361  -60.92122523]
 [ 182.27829083  116.9822964    85.07839589]
 [ -20.31785642

In [8]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.58112947856709

17O2 sigma:
 6.581129478567109

17O3 sigma:
 6.5811294785671866

17O4 sigma:
 6.581129478567174

17O5 sigma:
 8.184372724530347

17O6 sigma:
 8.184372724530336

17O7 sigma:
 8.184372724530363

17O8 sigma:
 8.184372724530354



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.93  -0.936  4.789]
 [-0.936 -0.743 -3.625]
 [ 4.789 -3.625 -0.187]]

CS Tensor:
 [[-76.302 169.627 -78.567]
 [171.694  74.019  27.976]
 [  9.917 -24.324 -58.703]]

CS isotropic Tensor:
 [[-20.329   0.      0.   ]
 [  0.    -20.329   0.   ]
 [  0.      0.    -20.329]]

CS symmetric Tensor:
 [[-76.302 170.66  -34.325]
 [170.66   74.019   1.826]
 [-34.325   1.826 -58.703]]

CS antisymmetric Tensor:
 [[  0.     -1.034 -44.242]
 [  1.034   0.     26.15 ]
 [ 44.242 -26.15    0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.58627275 -1.02695038 -5.55932237] 

 Unsorted Eigenvectors:
 [[-0.62830029  0.6114056  -0.48106335]
 [ 0.40786624  0.78543659  0.46554753]
 [-0.66248312 -0.09629415  0.74286174]] 

Sorted Eigenvalues: 
 [-1.02695038 -5.55932237  6.58627275] 

Sorted Eigenvectors: 
 [[ 0.6114056  -0.48106335 -0.62830029]
 [ 0.78543659  0.46554753  0.40786624]
 [-0.09629415  0.74286174 -0.66248312]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 186.55809944 -194.18893866  -53.3554156 ] 

 Unsorted Eigenvectors:
 [[ 0.54965974  0.82317342 -0.14233651]
 [ 0.83238907 -0.52525203  0.17674486]
 [-0.07072913  0.21562889  0.97391045]] 

Sorted Eigenvalues: 
 [ -53.3554156  -194.18893866  186.55809944] 

Sorted Eigenvectors: 
 [[-0.14233651  0.82317342  0.54965974]
 [ 0.17674486 -0.52525203  0.83238907]
 [ 0.97391045  0.21562889 -0.07072913]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -1.026950379927576 -5.559322369421468 6.5862727493491455
CSA Tensor Components δyy, δxx, δzz: 
 -53.35541559954374 -194.18893865829594 186.5580994370206


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   6.58627  |
+--------------+------------+
| etaq         |   0.688154 |
+--------------+------------+
| iso_cs (ppm) | -20.3288   |
+--------------+------------+
| csa (ppm)    | 206.887    |
+--------------+------------+
| etas         |   0.680727 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.48106335  0.6114056  -0.62830029]
 [ 0.46554753  0.78543659  0.40786624]
 [ 0.74286174 -0.09629415 -0.66248312]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-7.385834961357253 131.48952544940022 32.98994545943178 

Direction cosine csa: 

[[ 0.82317342 -0.14233651  0.54965974]
 [-0.52525203  0.17674486  0.83238907]
 [ 0.21562889  0.97391045 -0.07072913]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
77.5158059091979 94.05586730509233 -56.561624029236775 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 28.952745781976404 chi: 87.64970501388296 xi: -85.95080610488326 

